# 03 - Augmentasi & Penyetaraan Sampel

Menyetarakan **train** menjadi **3.000 gambar per kelas** (dataset gabungan GCD+CCSN): kelas > 3.000 di-*trim* (subsample acak), kelas < 3.000 diperbanyak dengan **augmentasi**. **Hanya train**; validation dan test tidak disentuh. Metodologi: [`docs/PENYETARAAN_SAMPEL.md`](../docs/PENYETARAAN_SAMPEL.md).

Teknik augmentasi (Pillow): translasi, flip horizontal, rotasi, color, contrast, brightness. Pendekatan di sini **pra-bangkitkan ke disk** (Opsi A) ke `data/combined_balanced/train/<kelas>/`. Alternatif on-the-fly saat training (Opsi B) tidak menulis berkas.

**Sumber:** `data/combined/train/` (hasil `scripts/materialize_combined.py`). `data/raw/` tidak disentuh.

## 1. Setup

In [1]:
import shutil
from pathlib import Path
import random

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
SEED = 42
TARGET_PER_CLASS = 3000
SRC = PROJECT_DIR / "data" / "combined" / "train"           # train gabungan termaterialisasi
DEST = PROJECT_DIR / "data" / "combined_balanced" / "train"  # keluaran seimbang
TABLE_DIR = PROJECT_DIR / "reports" / "tables"
assert SRC.is_dir(), f"Sumber tidak ada: {SRC}. Jalankan scripts/materialize_combined.py dulu."

classes = sorted(d.name for d in SRC.iterdir() if d.is_dir())
counts = {c: len(list((SRC / c).glob("*.jpg"))) for c in classes}
print("Sumber:", SRC.relative_to(PROJECT_DIR).as_posix())
print("Target per kelas:", TARGET_PER_CLASS, "| seed:", SEED)
for c in classes:
    print(f"  {c:16s} {counts[c]:>5}")
print("  TOTAL           ", sum(counts.values()))

Sumber: data/combined/train
Target per kelas: 3000 | seed: 42
  1_cumulus          809
  2_altocumulus     1027
  3_cirrus          1511
  4_clearsky        1669
  5_stratocumulus   2187
  6_cumulonimbus    2986
  7_mixed            294
  TOTAL            10483


## 2. Fungsi augmentasi

Tiap citra augmentasi memakai kombinasi acak: flip horizontal (p=0.5), rotasi ±15°, translasi ±10%, lalu color/contrast/brightness dengan faktor acak. Label tidak berubah.

In [2]:
def aug_translate(im, rng, frac=0.10):
    dx = rng.randint(-int(im.width * frac), int(im.width * frac))
    dy = rng.randint(-int(im.height * frac), int(im.height * frac))
    return im.rotate(0, translate=(dx, dy), fillcolor=(0, 0, 0))

def augment(im, rng):
    x = im
    if rng.random() < 0.5:
        x = ImageOps.mirror(x)                                   # flip horizontal
    x = x.rotate(rng.uniform(-15, 15), fillcolor=(0, 0, 0), resample=Image.BILINEAR)  # rotasi
    x = aug_translate(x, rng)                                     # translasi
    x = ImageEnhance.Color(x).enhance(rng.uniform(0.8, 1.5))      # color
    x = ImageEnhance.Contrast(x).enhance(rng.uniform(0.8, 1.4))   # contrast
    x = ImageEnhance.Brightness(x).enhance(rng.uniform(0.8, 1.3)) # brightness
    return x

print("Fungsi augmentasi siap: translasi, flip-H, rotasi, color, contrast, brightness.")

Fungsi augmentasi siap: translasi, flip-H, rotasi, color, contrast, brightness.


## 3. Rencana penyetaraan (target 3.000/kelas)

In [3]:
plan = []
for c in classes:
    n = counts[c]
    if n > TARGET_PER_CLASS:
        aksi, delta = "trim", n - TARGET_PER_CLASS
    elif n < TARGET_PER_CLASS:
        aksi, delta = "augmentasi", TARGET_PER_CLASS - n
    else:
        aksi, delta = "tetap", 0
    plan.append({"kelas": c, "sekarang": n, "aksi": aksi, "delta": delta,
                 "faktor": round(TARGET_PER_CLASS / n, 2), "target": TARGET_PER_CLASS})
plan = pd.DataFrame(plan)
print(plan.to_string(index=False))
print(f"\nTotal train sesudah: {TARGET_PER_CLASS * len(classes):,} ({len(classes)} kelas x {TARGET_PER_CLASS})")

          kelas  sekarang       aksi  delta  faktor  target
      1_cumulus       809 augmentasi   2191    3.71    3000
  2_altocumulus      1027 augmentasi   1973    2.92    3000
       3_cirrus      1511 augmentasi   1489    1.99    3000
     4_clearsky      1669 augmentasi   1331    1.80    3000
5_stratocumulus      2187 augmentasi    813    1.37    3000
 6_cumulonimbus      2986 augmentasi     14    1.00    3000
        7_mixed       294 augmentasi   2706   10.20    3000

Total train sesudah: 21,000 (7 kelas x 3000)


## 4. Materialisasi train seimbang

Untuk tiap kelas: salin citra asli, lalu (a) bila kurang dari target, bangkitkan augmentasi sampai target; (b) bila lebih, subsample acak berseed. Keluaran ke `data/combined_balanced/train/<kelas>/` dan manifest ke `reports/tables/combined_balanced_train_manifest.csv`.

In [4]:
if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir(parents=True, exist_ok=True)

rows = []
for c in classes:
    rng = random.Random(SEED + hash(c) % 10000)
    src_files = sorted((SRC / c).glob("*.jpg"))
    out_dir = DEST / c
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(src_files) > TARGET_PER_CLASS:                 # trim
        keep = rng.sample(src_files, TARGET_PER_CLASS)
        for p in keep:
            shutil.copy2(p, out_dir / p.name)
            rows.append({"class": c, "filename": p.name, "kind": "original"})
    else:                                                 # salin asli + augmentasi
        for p in src_files:
            shutil.copy2(p, out_dir / p.name)
            rows.append({"class": c, "filename": p.name, "kind": "original"})
        need = TARGET_PER_CLASS - len(src_files)
        for i in range(need):
            src = src_files[i % len(src_files)]
            with Image.open(src) as im:
                out = augment(im.convert("RGB"), rng)
            fn = f"{src.stem}_aug{i:05d}.jpg"
            out.save(out_dir / fn, quality=95)
            rows.append({"class": c, "filename": fn, "kind": "augmented"})
    print(f"  {c:16s} -> {len(list(out_dir.glob('*.jpg'))):>5} citra")

manifest = pd.DataFrame(rows)
manifest["split"] = "train"
manifest["rel_path"] = "train/" + manifest["class"] + "/" + manifest["filename"]
MAN = TABLE_DIR / "combined_balanced_train_manifest.csv"
manifest.to_csv(MAN, index=False, encoding="utf-8")
print("\nManifest:", MAN.relative_to(PROJECT_DIR).as_posix(), "| baris:", len(manifest))

  1_cumulus        ->  3000 citra
  2_altocumulus    ->  3000 citra
  3_cirrus         ->  3000 citra
  4_clearsky       ->  3000 citra
  5_stratocumulus  ->  3000 citra
  6_cumulonimbus   ->  3000 citra
  7_mixed          ->  3000 citra

Manifest: reports/tables/combined_balanced_train_manifest.csv | baris: 21000


## 5. Verifikasi

In [5]:
on_disk = {c: len(list((DEST / c).glob("*.jpg"))) for c in classes}
tab = manifest.groupby(["class", "kind"]).size().unstack(fill_value=0)
tab["total"] = tab.sum(axis=1)
tab["di_disk"] = pd.Series(on_disk)
tab.loc["TOTAL"] = tab.sum()
print(tab.to_string())
assert all(on_disk[c] == TARGET_PER_CLASS for c in classes), "Ada kelas yang tidak tepat 3.000."
assert sum(on_disk.values()) == TARGET_PER_CLASS * len(classes)
print(f"\nOK: semua kelas = {TARGET_PER_CLASS}; total train seimbang = {sum(on_disk.values()):,}.")

kind             augmented  original  total  di_disk
class                                               
1_cumulus             2191       809   3000     3000
2_altocumulus         1973      1027   3000     3000
3_cirrus              1489      1511   3000     3000
4_clearsky            1331      1669   3000     3000
5_stratocumulus        813      2187   3000     3000
6_cumulonimbus          14      2986   3000     3000
7_mixed               2706       294   3000     3000
TOTAL                10517     10483  21000    21000

OK: semua kelas = 3000; total train seimbang = 21,000.


## 6. Contoh hasil augmentasi

In [6]:
import matplotlib
import matplotlib.pyplot as plt

cls = "7_mixed"                                   # kelas minoritas (paling banyak diaugmentasi)
aug_files = sorted((DEST / cls).glob("*_aug*.jpg"))[:5]
orig = sorted(p for p in (DEST / cls).glob("*.jpg") if "_aug" not in p.name)[0]
show = [("asli", orig)] + [("augmentasi", p) for p in aug_files]

fig, axes = plt.subplots(1, len(show), figsize=(2.2 * len(show), 2.4))
for ax, (tag, p) in zip(axes, show):
    with Image.open(p) as im:
        ax.imshow(im)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"{tag}\n{p.name}", fontsize=7)
fig.suptitle(f"{cls}: contoh augmentasi (hanya train)", fontsize=10)
fig.tight_layout()
plt.show()

<Figure size 1320x240 with 6 Axes>

## 7. Hasil & catatan

- **Train seimbang**: `data/combined_balanced/train/<kelas>/` — **21.000 citra** (7 × 3.000). Manifest: `reports/tables/combined_balanced_train_manifest.csv` (kolom `class`, `kind` = original/augmented, `rel_path`).
- **Trimming tidak aktif** untuk train ini (semua kelas < 3.000); semua kelas diaugmentasi naik.
- **Validation & test tidak diaugmentasi** — tetap `data/combined/val` (1.851) dan `data/combined/test` (9.000); distribusi nyata dipertahankan untuk evaluasi yang jujur.
- Augmentasi warna dijaga tidak ekstrem; `7_mixed` diperbanyak ~10x (risiko overfitting — lihat `docs/PENYETARAAN_SAMPEL.md` 4.3).
- **Alternatif** on-the-fly (Opsi B): `WeightedRandomSampler` + augmentasi acak tiap epoch saat training, tanpa menulis berkas.
- `data/raw/` tidak diubah; `data/` tidak dilacak Git.